In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.ingestion.web_loader import load_website

In [ ]:
url = "https://www.geeksforgeeks.org"

documents = load_website(url)
print(documents[0].page_content[:1000])


In [ ]:
print(documents[0].metadata)

In [ ]:
from src.ingestion.youtube_loader import load_youtube


In [ ]:
youtube_url = "https://youtu.be/y48WlxiLlNY"
documents = load_youtube(youtube_url)

print(type(documents))
print("Number of documents:", len(documents))
print(documents[0].page_content[:1000])

In [ ]:
youtube_url = "https://youtu.be/t_qSRhiHf2w?list=RD6guo9iOCnus"

documents = load_youtube(youtube_url)

print(type(documents))
print("Number of documents:", len(documents))
print(documents[0].page_content[:1000])

In [ ]:
youtube_url = "https://youtu.be/JJWE3QH-Q6s?list=RDB78jVe9mTCg"

documents = load_youtube(youtube_url)

print(type(documents))
print("Number of documents:", len(documents))
print("Language:", documents[0].metadata["language"])
print("Language code:", documents[0].metadata["language_code"])
print()
print(documents[0].page_content[:1000])

In [ ]:
from src.ingestion.pdf_loader import load_pdf

pdf_path = r"C:\Users\maury\Downloads\Linear Regression.pdf"

documents = load_pdf(pdf_path)

print("Number of pages:", len(documents))

In [ ]:
print(documents[0].page_content[:2000])

In [ ]:
print(documents[0].metadata)

In [ ]:
from src.ingestion.docx_loader import load_docx

In [ ]:
docx_path = r"C:\Users\maury\Downloads\sample-document-small.docx"

documents = load_docx(docx_path)

print("Number of documents:", len(documents))

for i, document in enumerate(documents[:5]):
    print(f"\n--- Document {i + 1} ---")
    print(document.page_content)

In [ ]:
from src.processing.text_processor import split_documents

In [ ]:
chunks = split_documents(documents)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content[:500])

In [ ]:
from src.embeddings.embedding_service import (
    get_embedding_model,
    generate_embeddings
)

embedding_model = get_embedding_model()

In [ ]:
sample_texts = [
    "Machine learning enables computers to learn from data.",
    "Deep learning uses neural networks.",
    "The weather is sunny today."
]

vectors = generate_embeddings(
    sample_texts,
    embedding_model
)

print("Number of vectors:", len(vectors))
print("Vector dimensions:", len(vectors[0]))
print("First vector sample:", vectors[0][:10])

In [ ]:
query = "How do computers learn from data?"

query_vector = embedding_model.embed_query(query)

print("Query vector dimensions:", len(query_vector))

In [ ]:

chunk_texts = [
    chunk.page_content for chunk in chunks
]

chunk_vectors = generate_embeddings(
    chunk_texts,
    embedding_model
)

print("Total chunks:", len(chunks))
print("Total embeddings:", len(chunk_vectors))
print("Embedding dimensions:", len(chunk_vectors[0]))

In [ ]:
print("First chunk metadata:", chunks[0].metadata)
print("First chunk text:", chunks[0].page_content[:300])

In [ ]:
from src.retrieval.vector_store import create_vector_store

In [ ]:
vector_store = create_vector_store(
    chunks,
    embedding_model
)

In [ ]:
query = "What is machine learning?"

results = vector_store.similarity_search(
    query,
    k=3
)

print("Number of results:", len(results))

for i, result in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(result.page_content[:500])
    print("Metadata:", result.metadata)

In [ ]:
from src.retrieval.retriever import create_retriever

retriever = create_retriever(
    vector_store,
    k=3
)


In [ ]:
query = "What is machine learning?"

retrieved_documents = retriever.invoke(query)

print("Number of retrieved documents:", len(retrieved_documents))

for i, document in enumerate(retrieved_documents):
    print(f"\n--- Retrieved Document {i + 1} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)

In [ ]:
from src.rag.qa_chain import answer_question

In [ ]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY was not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=groq_api_key,
    temperature=0
)


In [ ]:
question = "What is machine learning?"

result = answer_question(
    question,
    retriever,
    llm
)

print("Answer:")
print(result["answer"])